<center>
    <p style="text-align:center">
        <img alt="phoenix logo" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://docs.arize.com/phoenix/">Docs</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">Community</a>
    </p>
</center>

# Langchain Annotations

In this tutorial, we'll explore how to add custom annotations to your Phoenix traces, specifically when working with langchain.


In [ ]:
!pip install openai \
  opentelemetry-api \
  opentelemetry-sdk \
  openinference-semantic-conventions \
  openinference-instrumentation-openai \
  openinference-instrumentation-langchain \
  opentelemetry-exporter-otlp \
  arize-phoenix \
  langgraph \
  "langchain[openai]" \
  'httpx<0.28'

In [ ]:
import os
from getpass import getpass

from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry import trace as trace_api
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk import trace as trace_sdk
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

In [ ]:
# setup tracing

resource = Resource(attributes={})
tracer_provider = trace_sdk.TracerProvider(resource=resource)
span_exporter = OTLPSpanExporter(endpoint="http://localhost:6006/v1/traces")
span_processor = SimpleSpanProcessor(span_exporter=span_exporter)
tracer_provider.add_span_processor(span_processor=span_processor)
trace_api.set_tracer_provider(tracer_provider=tracer_provider)

tracer = trace_api.get_tracer(__name__)
OpenAIInstrumentor().instrument(skip_dep_check=True)

In [ ]:
import phoenix as px

px.launch_app().view()

In [ ]:
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
llm = init_chat_model("openai:gpt-4.1")

In [ ]:
# simple langgraph graph

from typing import Annotated

from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict


class State(TypedDict):
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)


def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}


graph_builder.add_node("chatbot", chatbot)

graph_builder.add_edge(START, "chatbot")

graph = graph_builder.compile()

from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
def stream_queries(user_input: str):
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

In [ ]:
# streaming queries and retrieving the current span for each query

from opentelemetry.trace import format_span_id, get_current_span

with tracer.start_as_current_span("span") as span:
    # OpenTelemetry get_current_span
    span = get_current_span()
    if span is not None:
        span_id = format_span_id(span.get_span_context().span_id)
    print(span)

    while True:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_queries(user_input)

In [ ]:
from phoenix.client import Client

client = Client()
annotation = client.annotations.add_span_annotation(
    annotation_name="user feedback",
    annotator_kind="HUMAN",
    span_id=span_id,
    label="thumbs-up",
    score=1,
)